# GBFS Audit Catalogue — Experiment Reproduction Notebook

**For reviewers and replicators.** This notebook reproduces the key results of experiments XP2 (topology-aware A4 ablation) and XP3 (leave-one-operator-out cross-validation) reported in the manuscript, directly from the certified catalogue parquet.

**Requirements:** `pip install -e ".[experiments]"` from the repository root.

| Experiment | Section in paper | What it demonstrates |
|---|---|---|
| **XP2** | §2.5 (Ablation), §4.5 (A4 v2) | Composite detector eliminates 8,005 legacy false positives |
| **XP3** | §2.6 (LOOO) | Rules are not overfit to specific operators |
| **XP1** | §3.2 (Limitation L2) | Protocol ready; requires 14-day data collection |

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from audit_pipeline.core import load_catalogue, enrich

df = load_catalogue()
print(f"Catalogue: {len(df):,} stations, {df['system_id'].nunique()} systems")
df.head(3)

## XP2 — Topology-aware A4 ablation (Table 4, Figure 9)

Runs the HDBSCAN + spectral composite detector alongside the legacy centroid + 3σ MAD, then classifies each station into one of four discordance classes.

In [ ]:
from experiments.xp2_spatial_topology.ablation import run_ablation, ablation_summary, geometry_type_heuristic

ablation_df = run_ablation(df)
print("=== Discordance distribution ===")
print(ablation_df["discordance_class"].value_counts())
print(f"\nTotal: {len(ablation_df):,} stations")

In [ ]:
# Top-10 systems with highest discordance rate
abl_sum = ablation_summary(ablation_df)
print("=== Top-10 systems by discordance rate ===")
abl_sum.head(10)[["system_id", "n_stations", "n_fp_legacy", "n_fn_composite", "discordance_rate"]]

In [ ]:
# Geometry classification and FP_LEGACY concentration
geo_df = geometry_type_heuristic(df)
merged = ablation_df.merge(geo_df, on="system_id", how="left")
fp = merged[merged["discordance_class"] == "FP_LEGACY"]

print("=== FP_LEGACY by geometry type ===")
print(fp["geometry_type"].value_counts())
print(f"\nLinear systems account for {fp[fp['geometry_type']=='linear'].shape[0]:,} / {len(fp):,} false positives "
      f"({fp[fp['geometry_type']=='linear'].shape[0]/len(fp)*100:.1f}%)")

## XP3 — Leave-One-Operator-Out cross-validation (Table 5, Figure 10)

For each of the 7 eligible operators (≥50 stations, ≥2 systems), train rule thresholds on all other operators, apply to the held-out operator, and measure flag-rate stability.

In [ ]:
df_enriched = enrich(df)

from experiments.xp3_looo_validation.protocol import run_looo_full, summarise_looo, RULE_COLS

fold_results = run_looo_full(df_enriched, min_stations_per_operator=50, min_systems_per_operator=2)
summary = summarise_looo(fold_results, n_bootstrap=500)

print(f"=== LOOO: {len(fold_results)} folds ===\n")
for rule in RULE_COLS:
    cv = summary.per_rule_cv[rule]
    ci = summary.bootstrap_ci[rule]
    verdict = "PASS" if cv < 0.20 else "FAIL"
    print(f"  {rule}: CV={cv:.3f} [{verdict}], rate={ci['mean']:.4f} [{ci['ci_lo']:.4f}, {ci['ci_hi']:.4f}]")

In [ ]:
# Per-operator flag rate heatmap
print("=== Per-operator test-fold flag rates ===")
summary.summary_df[["operator", "n_test"] + [f"{r}_rate_test" for r in RULE_COLS]].round(3)

In [ ]:
# H3 validation: negative controls (clean dock-based operators)
print("=== H3 validation: clean operators as negative controls ===\n")
for op in ["Vélib' Métropole", "Vélo&Co"]:
    fold = [r for r in fold_results if r.held_out_operator == op]
    if fold:
        r = fold[0]
        print(f"{op} ({r.n_test} stations):")
        for rule in RULE_COLS:
            rate = r.flag_rates_test[rule]
            print(f"  {rule}: {rate*100:.1f}%")
        print()

## XP1 — Dynamic audit protocol (ready, not yet executed)

The code below shows the classification logic on **synthetic data** to demonstrate the entropy-based zombie detection. A real 14-day collection run is needed for production results.

The four liveness classes are:
- **LIVE**: H(s) ≥ 0.01, significant temporal variance
- **ZOMBIE**: H(s) < 0.01 AND last_reported stale > 72h
- **INTERMITTENT**: frozen fraction > 0.90 but H(s) ≥ 0.01
- **DECOMMISSIONED**: present in station_information, absent from station_status

In [ ]:
from datetime import datetime, timezone
from experiments.xp1_dynamic_audit.detector import classify_stations, _normalised_entropy

# Demonstrate entropy on synthetic series
rng = np.random.default_rng(42)
live_series = rng.integers(0, 20, size=200).astype(float)
zombie_series = np.full(200, 5.0)
intermittent_series = np.concatenate([np.full(180, 5.0), rng.integers(0, 20, size=20)]).astype(float)

print(f"Live station entropy:         H = {_normalised_entropy(live_series):.4f}")
print(f"Zombie station entropy:       H = {_normalised_entropy(zombie_series):.4f}")
print(f"Intermittent station entropy:  H = {_normalised_entropy(intermittent_series):.4f}")
print(f"\nThreshold: H < 0.01 => ZOMBIE candidate")